# Sentiment Analysis

In [53]:
import os
print(os.path.abspath("arg_mining/ml_algorithms/ML/datasets/test.conll"))

/Users/chamroeunphal/Desktop/mining_proj/arg_mining/ml_algorithms/ML/arg_mining/ml_algorithms/ML/datasets/test.conll


In [54]:
train_path = "datasets/train.conll"
test_path = "datasets/test.conll"
val_path = "datasets/validation.conll"
with open(train_path, encoding="utf-8") as f:
    train_data = f.read()

with open(test_path, encoding = "utf-8") as f:
    test_data = f.read()

with open(val_path, encoding = "utf-8") as f:
    val_data = f.read()

In [55]:
#extract first column directly from conll files
def get_first_column(data):
    lines = data.strip().split('\n')
    first_col = []

    for i in lines:
        if i.strip():
            cols = i.split('\t')
            first_col.append(cols[0])

    return first_col

train_first_col = get_first_column(train_data)
test_first_col = get_first_column(test_data)
val_first_col = get_first_column(val_data)

len(train_first_col), len(test_first_col), len(val_first_col)


(943056, 236923, 1179979)

In [56]:
#join words to form sentence separated by tab
'''
open the conll file then remove whitespace, tabs, newline
there is only 1 column in the conll files so it'll just append those words into current_sentence
then join the current word to another word below to make a sentence
the final else indicates the last word of a sentence
'''
def read_conll_file(file_path):
    sentences = []
    current_sentence = []
    with open(file_path, encoding='utf-8') as f:
        for line in f:
            line = line.lower()
            line = line.strip()
            if line:
                parts = line.split('\t')
                word = parts[0]
                current_sentence.append(word)
            else:
                if current_sentence:
                    sentences.append(' '.join(current_sentence))
                    current_sentence = []

    if current_sentence:
        sentences.append(' '.join(current_sentence))
    return sentences

train_sentences = read_conll_file("datasets/train.conll")
test_sentences = read_conll_file("datasets/test.conll")
val_sentences = read_conll_file("datasets/validation.conll")
test_sentences[:10]

['comment i live in an agricultural area of my state i am appreciative of alllllll the crap alllll the farmers ranchers and producers deal with its alot',
 'generally american cities have a lot of really awesome neighborhoods the transit to get from one to the other is the commonly missing part and also lots of americans live in the suburbs which often wont have nice neighborhoods even',
 'comment americanflagscom some they are kind of pricy but were running a sale soon',
 '14 ronmckelvey',
 'there is something wrong with our culture and we need some serious introspection',
 'comment removed',
 'comment my sister had the knack for finding 4 leaf clovers she would find multiples all the time her husband worked a dangerous job and wore a 4 leaf clover for luck when the clover would wilt she would just go and find another no problem',
 '19 nkpstudios',
 'comment this is a subreddit for genuine discussion',
 '9 tiger0204']

In [57]:
#remove reddit usernames (elements start with a number)
train_sentences = [item for item in train_sentences if not item.split()[0].isdigit()]
test_sentences = [item for item in test_sentences if not item.split()[0].isdigit()]
val_sentences = [item for item in val_sentences if not item.split()[0].isdigit()]
train_sentences[:5]

['comment my grandparents greatestwhatever was before greatest generation used the gesture as well as regularly saying shame on you or you should be ashamed of yourself pretty often and it was very much considered the appropriate parenting stylei read a lot of early 20th century and late 19th century popular lit kids books dime novels magazine fiction and there were frequent scenes with similar language as an older gen x i heard it from grandparents and teachers and nuns but less from my parents silents as i think they had started to recognize that excessive guilt and shame is harmful',
 'comment its grey and cloudy today anyway',
 'comment i assume you meant greet',
 'violators will be fed to the bear',
 'i am not a personal fan of it on myself i literally dont care about it on anyone else though']

In [58]:
#vocab to keep
negation_words = [
    "not", "no", "never", "none", "nothing", "neither", "nor",
    "hardly", "scarcely", "barely", "without"
]
intensifiers = [
    "very", "really", "extremely", "quite", "so", "too", "just",
    "absolutely", "totally", "incredibly", "barely", "fairly", "almost", "nearly"
]
modal_verbs = [
    "could", "would", "should", "might", "may", "must", "can", "shall", "will"
]
auxiliary_verbs = [
    "is", "are", "was", "were", "be", "been", "being", "am",
    "do", "does", "did", "have", "has", "had"
]
pronouns = [
    "i", "you", "we", "they", "he", "she", "it",
    "me", "us", "them", "my", "your", "our", "their",
    "mine", "yours", "his", "hers", "its"
]
conjunctions = [
    "but", "although", "though", "yet", "while", "whereas"
]
subjective_adverbs = [
    "always", "never", "sometimes", "often", "seldom",
    "unfortunately", "fortunately", "luckily", "sadly", "happily"
]
exception_words = (
    negation_words +
    intensifiers +
    modal_verbs +
    auxiliary_verbs +
    pronouns +
    conjunctions +
    subjective_adverbs
)
exception_words[:10]

['not',
 'no',
 'never',
 'none',
 'nothing',
 'neither',
 'nor',
 'hardly',
 'scarcely',
 'barely']

### Preprocess Text

1. source text
* data cleaning
    * identify noise
    * noise removal
    * character normalization
    * data masking*
* linguistic processing
    * tokenization
    * POS tagging
    * stopwords
    * lemmatization
    * named-entity recognition
        

In [59]:
#nltk.download('all')
import pandas as pd
import nltk 
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords 
from nltk.tokenize import word_tokenize 
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text)
    filtered_tokens = [token for token in tokens if token.lower() not in stop_words]
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    return " ".join(lemmatized_tokens)

train_processed = [preprocess_text(sentence) for sentence in train_sentences]
test_processed = [preprocess_text(sentence) for sentence in test_sentences]
val_processed = [preprocess_text(sentence) for sentence in val_sentences]

len(train_processed), len(test_processed), len(val_processed)

(29517, 7415, 36932)

In [60]:
#turn to pandas dataframe
df_train = pd.DataFrame({'sentences': train_processed})
df_test = pd.DataFrame({'sentences': test_processed})
df_val = pd.DataFrame({'sentences': val_processed})

In [61]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    scores = analyzer.polarity_scores(text)
    if scores['pos'] > 0:
        sentiment = 1
    else:
        sentiment = 0
    return sentiment 

df_train['sentiment'] = df_train['sentences'].apply(get_sentiment)
df_test['sentiment'] = df_test['sentences'].apply(get_sentiment)
df_val['sentiment'] = df_val['sentences'].apply(get_sentiment)
    

In [62]:
from nltk.corpus import opinion_lexicon
negative_words = list(opinion_lexicon.negative())
negative_words[:10]

['2-faced',
 '2-faces',
 'abnormal',
 'abolish',
 'abominable',
 'abominably',
 'abominate',
 'abomination',
 'abort',
 'aborted']

In [63]:
#make a new column that shows positive sentence 
#expand contractions

%pip install contractions
%pip install swifter
import swifter
import contractions

negative_words = set(opinion_lexicon.negative())

def set_positive(sentence):
    expanded = contractions.fix(sentence)
    words = word_tokenize(expanded.lower())
    for i in words:
        if i in negative_words:
            return 0
    return 1

df_train['positive'] = df_train['sentences'].swifter.apply(set_positive)
df_test['positive'] = df_test['sentences'].swifter.apply(set_positive)
df_val['positive'] = df_val['sentences'].swifter.apply(set_positive)
    

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Pandas Apply:   0%|          | 0/29517 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/7415 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/36932 [00:00<?, ?it/s]

In [64]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(df_train['positive'], df_train['sentiment']))

[[ 3935 10234]
 [ 7909  7439]]


Confusion matrix: *many errors*

In [65]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(df_train['sentences'])
y_train = df_train['sentiment']

In [66]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

random_forest_classi = RandomForestClassifier(n_estimators=100, random_state = 42)
random_forest_classi.fit(X_train, y_train)

y_pred = random_forest_classi.predict(X_train)

conf_matrix = confusion_matrix(y_train, y_pred)
print("Confusion matrix: \n", conf_matrix)

print("\nClassification Report: \n", classification_report(y_train, y_pred))



Confusion matrix: 
 [[11844     0]
 [    0 17673]]

Classification Report: 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     11844
           1       1.00      1.00      1.00     17673

    accuracy                           1.00     29517
   macro avg       1.00      1.00      1.00     29517
weighted avg       1.00      1.00      1.00     29517



confusion matrix: *need to improve this unrealistic confusion matrix*